# APAR DataRace — Legal High-Ceiling Neural + Classical Pipeline

This notebook is designed for the **Apar - İstifadəçi Rəylərinin Təsnifatı** competition.

It follows the discussion/rule clarifications:

- ✅ Private Kaggle / Colab GPU is allowed if the data is not made public.
- ✅ Open-source local neural models are allowed if the final inference pipeline stays within the parameter limit.
- ✅ Knowledge distillation is allowed when the **test set is not used**.
- ✅ The 600M parameter limit is for the **final inference pipeline**.
- ✅ One final checkpoint below 600M is safe; k-fold inference ensembles count all checkpoints.
- ⚠️ Checkpoint/weight averaging should be treated as optional unless you are comfortable defending it as one final model.
- ❌ No Gemini/OpenAI/Claude API inference.
- ❌ No test-set MLM/domain-adaptation training.
- ❌ No pseudo-label training on `test.csv`.
- ❌ Do not manually/AI-label test rows as training data.

Main idea:

1. Fine-tune a **single multilingual transformer under 600M parameters**.
2. Use the `tag` column as text prefix.
3. Optimize Macro-F1 using out-of-fold validation.
4. Apply **train-only** calibration/threshold biases.
5. Optionally blend with a classical TF-IDF model only as a small local component.
6. Generate valid `id,label` submissions.

Recommended starting model: `xlm-roberta-base` (~270M params), fast and multilingual.
Optional high-ceiling model: `FacebookAI/xlm-roberta-large` (~559M params), slower and close to the limit.

## 0. Install packages

Run this in Kaggle private notebook or Colab. On Kaggle, enable GPU.

In [ ]:
import sys, subprocess, os

def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

try:
    import transformers, torch, sklearn
    print('transformers', transformers.__version__)
    print('torch', torch.__version__)
except Exception:
    pip_install(['transformers>=4.41.0', 'accelerate>=0.30.0', 'sentencepiece', 'safetensors', 'joblib', 'tqdm'])

## 1. Imports and deterministic config

In [ ]:
import os, re, gc, math, json, random, shutil, inspect
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from scipy.special import softmax
from scipy.optimize import differential_evolution
from scipy import sparse

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))

## 2. Configuration

Choose one model. `xlm-roberta-base` is the safest default under 600M. `FacebookAI/xlm-roberta-large` may score higher but is slower and close to the limit.

In [ ]:
CFG = {
    # Good safe default: ~270M params
    'model_name': 'xlm-roberta-base',

    # Optional high-ceiling model; use only one neural model in final inference.
    # 'model_name': 'FacebookAI/xlm-roberta-large',

    'max_length': 96,
    'n_folds': 5,
    'run_cv': True,          # Set False for quick final-only training.
    'epochs': 4,
    'batch_size': 16,        # For xlm-roberta-large try 4 or 8.
    'grad_accum': 1,
    'lr': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.08,
    'fp16': True,
    'output_dir': './apar_neural_outputs',
    'disable_checkpoint_saving': True,  # prevents Kaggle disk from filling during CV
    'save_final_model': False,          # predictions are generated in-memory; set True only if you have disk space
    'use_class_weights': True,

    # Final training
    'final_epochs': 5,
    'final_lr': 1.5e-5,

    # Optional checkpoint soup: train several full-data runs, average weights into one final checkpoint.
    # Use only if you are comfortable defending final inference as a single model.
    'run_weight_soup': False,
    'soup_seeds': [42, 777, 2026],
}
Path(CFG['output_dir']).mkdir(exist_ok=True, parents=True)
print(json.dumps(CFG, indent=2))

In [ ]:
# Emergency disk cleanup for Kaggle: run this before training if you saw
# `SafetensorError: No space left on device`.
import shutil, os, glob
for p in [CFG['output_dir'], '/kaggle/working/apar_neural_outputs', '/kaggle/temp/apar_neural_outputs']:
    if os.path.exists(p):
        print('Removing old output folder:', p)
        shutil.rmtree(p, ignore_errors=True)
Path(CFG['output_dir']).mkdir(exist_ok=True, parents=True)

# Optional: show disk status.
try:
    !df -h /kaggle/working /kaggle/temp
except Exception:
    pass


## 3. Load data

Works with files in `/content`, current directory, Kaggle input folders, or manual upload in Colab.

In [ ]:
def find_file(name):
    candidates = [Path(name), Path('/content')/name, Path('/kaggle/working')/name]
    candidates += list(Path('/kaggle/input').glob(f'**/{name}')) if Path('/kaggle/input').exists() else []
    for p in candidates:
        if p.exists():
            return p
    return None

for fname in ['train.csv', 'test.csv', 'sample.csv']:
    if find_file(fname) is None:
        print(f'{fname} not found.')

if any(find_file(x) is None for x in ['train.csv', 'test.csv', 'sample.csv']):
    try:
        from google.colab import files
        print('Please upload train.csv, test.csv, sample.csv')
        uploaded = files.upload()
    except Exception as e:
        print('Upload utility unavailable. Put train.csv, test.csv, sample.csv in working directory.')

train_path, test_path, sample_path = map(find_file, ['train.csv', 'test.csv', 'sample.csv'])
print(train_path, test_path, sample_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
print(train.shape, test.shape, sample.shape)
print(train.head())

## 4. Validation and label setup

In [ ]:
required_train = {'id', 'feedback', 'tag', 'label'}
required_test = {'id', 'feedback', 'tag'}
assert required_train.issubset(train.columns), train.columns
assert required_test.issubset(test.columns), test.columns
assert list(sample.columns) == ['id', 'label'], sample.columns
assert len(test) == len(sample)
assert test['id'].tolist() == sample['id'].tolist(), 'test/sample id order mismatch'

LABELS = ['technical_support', 'customer_support', 'other']
assert set(train['label'].unique()) <= set(LABELS)
label2id = {lab:i for i, lab in enumerate(LABELS)}
id2label = {i:lab for lab, i in label2id.items()}
train['y'] = train['label'].map(label2id).astype(int)
print(train['label'].value_counts())
print(train['tag'].fillna('NO_TAG').value_counts().head(30))

## 5. Text/tag preprocessing

Do not over-clean multilingual noisy text. We preserve Azerbaijani/Russian/English characters, punctuation, and emoji-like symbols.

In [ ]:
_ws = re.compile(r'\s+')

def normalize_tag(x):
    if pd.isna(x) or str(x).strip() == '':
        return 'NO_TAG'
    s = str(x).strip()
    s = s.replace('/', '_').replace(' ', '_').replace('-', '_')
    s = re.sub(r'[^\wа-яА-ЯəƏöÖüÜıİğĞçÇşŞ_]+', '', s)
    return s if s else 'NO_TAG'

def normalize_feedback(x):
    if pd.isna(x):
        return ''
    s = str(x)
    s = s.replace('\u200b', ' ').replace('\xa0', ' ')
    s = _ws.sub(' ', s).strip()
    # soft lower; multilingual safe enough
    return s.lower()

MONEY_TERMS = [
    'pul', 'ödəniş', 'odenis', 'ödeniş', 'payment', 'refund', 'geri qaytar', 'qaytarın',
    'kart', 'card', 'balance', 'balans', 'charged', 'charge edildi', 'деньги', 'оплата',
    'верните', 'возврат', 'списал', 'списали', 'сняли', 'карта', 'карты'
]

def make_text(feedback, tag):
    t = normalize_tag(tag)
    f = normalize_feedback(feedback)
    has_tag = 'HAS_TAG_0' if t == 'NO_TAG' else 'HAS_TAG_1'
    # Repeat tag prefix so model sees it even in short texts.
    return f'[TAG_{t}] [{has_tag}] {f}'

train['text'] = [make_text(f, t) for f, t in zip(train['feedback'], train['tag'])]
test['text'] = [make_text(f, t) for f, t in zip(test['feedback'], test['tag'])]
train[['feedback','tag','text','label']].sample(5, random_state=SEED)

## 6. Lightweight EDA and train-label noise hints

The discussions mention label noise and priority rules. We use this for model design, not for test labeling.

Important priority from organizer discussion: when a review mixes payment/refund/customer issue with technical issue, **customer_support > technical_support > other**.

In [ ]:
def has_money_issue(text):
    s = normalize_feedback(text)
    return any(term in s for term in MONEY_TERMS)

train['has_money_issue'] = train['feedback'].apply(has_money_issue)
print(pd.crosstab(train['has_money_issue'], train['label'], normalize='index'))
print(pd.crosstab(train['tag'].fillna('NO_TAG'), train['label']).sort_values('technical_support', ascending=False).head(20))

# Same normalized feedback appearing with multiple labels in train: label-noise indicator.
train['norm_feedback'] = train['feedback'].apply(normalize_feedback)
dup_label_counts = train.groupby('norm_feedback')['label'].nunique().sort_values(ascending=False)
print('Normalized feedbacks with multiple labels:', int((dup_label_counts > 1).sum()))

## 7. Dataset and trainer helpers

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=96):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
        )
        if self.labels is not None:
            enc['labels'] = int(self.labels[idx])
        return enc

class ClassWeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

def get_class_weights(y):
    counts = np.bincount(y, minlength=len(LABELS)).astype(float)
    weights = counts.sum() / (len(LABELS) * counts)
    # Slightly soften to avoid over-correcting majority/minority boundary.
    weights = np.sqrt(weights)
    return torch.tensor(weights, dtype=torch.float32)

class_weights = get_class_weights(train['y'].values) if CFG['use_class_weights'] else None
print('class_weights:', class_weights)

## 8. Train one neural fold/model

In [ ]:
def train_neural_model(train_texts, train_y, valid_texts=None, valid_y=None, out_dir='./model', seed=42, epochs=None, lr=None, save_model=None):
    """Train one model. By default this DOES NOT save checkpoints/models.

    Kaggle often crashes with: `No space left on device` while Trainer saves
    checkpoint shards. We avoid this by setting save_strategy='no' and by
    producing predictions from the in-memory model immediately after training.
    """
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'], use_fast=True)
    config = AutoConfig.from_pretrained(
        CFG['model_name'],
        num_labels=len(LABELS),
        id2label={str(i): lab for i, lab in id2label.items()},
        label2id=label2id,
    )
    model = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'], config=config)

    train_ds = TextDataset(train_texts, train_y, tokenizer, CFG['max_length'])
    eval_ds = None if valid_texts is None else TextDataset(valid_texts, valid_y, tokenizer, CFG['max_length'])
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    no_save = bool(CFG.get('disable_checkpoint_saving', True))
    args = TrainingArguments(
        output_dir=out_dir,
        seed=seed,
        num_train_epochs=epochs or CFG['epochs'],
        learning_rate=lr or CFG['lr'],
        per_device_train_batch_size=CFG['batch_size'],
        per_device_eval_batch_size=max(CFG['batch_size']*2, 8),
        gradient_accumulation_steps=CFG['grad_accum'],
        weight_decay=CFG['weight_decay'],
        # In newer transformers, warmup_ratio is deprecated; warmup_steps accepts a float ratio.
        warmup_steps=CFG['warmup_ratio'],
        fp16=bool(CFG['fp16'] and torch.cuda.is_available()),
        logging_steps=50,
        eval_strategy='epoch' if eval_ds is not None else 'no',
        save_strategy='no' if no_save else ('epoch' if eval_ds is not None else 'no'),
        save_total_limit=1,
        load_best_model_at_end=False if no_save else (True if eval_ds is not None else False),
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        report_to='none',
        dataloader_num_workers=2,
    )
    # Transformers API compatibility: newer versions removed `tokenizer=` from Trainer
    # and replaced it with `processing_class=`.
    trainer_kwargs = dict(
        class_weights=class_weights,
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        compute_metrics=compute_metrics if eval_ds is not None else None,
    )
    if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs['processing_class'] = tokenizer
    else:
        trainer_kwargs['tokenizer'] = tokenizer

    trainer = ClassWeightedTrainer(**trainer_kwargs)
    trainer.train()

    if save_model is None:
        save_model = bool(CFG.get('save_final_model', False))
    if save_model:
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        trainer.save_model(out_dir)
        tokenizer.save_pretrained(out_dir)

    return trainer, tokenizer


def predict_neural_in_memory(model, tokenizer, texts, batch_size=64):
    """Predict from the model already in memory; no disk load/save needed."""
    model = model.to(DEVICE)
    model.eval()
    ds = TextDataset(texts, None, tokenizer, CFG['max_length'])
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collator)
    logits_all = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            logits_all.append(out.logits.detach().cpu().numpy())
    return np.vstack(logits_all)


def predict_neural(model_dir, texts, batch_size=64):
    """Only use this if you intentionally saved a model to disk."""
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    logits = predict_neural_in_memory(model, tokenizer, texts, batch_size=batch_size)
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return logits


## 9. OOF CV for honest Macro-F1 and class-bias calibration

We use folds for **validation only**. Final inference will use one full-data model, not a 5-fold ensemble.

In [ ]:
oof_logits = np.zeros((len(train), len(LABELS)), dtype=np.float32)
fold_scores = []

if CFG['run_cv']:
    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(train['text'], train['y'])):
        print(f'\n===== Fold {fold+1}/{CFG["n_folds"]} =====')
        fold_dir = f"{CFG['output_dir']}/fold_{fold}"
        trainer, tok = train_neural_model(
            train.iloc[tr_idx]['text'].tolist(), train.iloc[tr_idx]['y'].tolist(),
            train.iloc[va_idx]['text'].tolist(), train.iloc[va_idx]['y'].tolist(),
            out_dir=fold_dir,
            seed=SEED + fold,
            epochs=CFG['epochs'],
            lr=CFG['lr'],
        )
        logits = predict_neural_in_memory(trainer.model, tok, train.iloc[va_idx]['text'].tolist(), batch_size=max(CFG['batch_size']*2, 16))
        oof_logits[va_idx] = logits
        pred = logits.argmax(1)
        score = f1_score(train.iloc[va_idx]['y'], pred, average='macro')
        fold_scores.append(score)
        print('Fold macro F1:', score)
        print(classification_report(train.iloc[va_idx]['y'], pred, target_names=LABELS))
        del trainer, tok
        if os.path.exists(fold_dir):
            shutil.rmtree(fold_dir, ignore_errors=True)
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print('OOF fold scores:', fold_scores)
    print('OOF mean/std:', float(np.mean(fold_scores)), float(np.std(fold_scores)))
    np.save(f"{CFG['output_dir']}/oof_logits.npy", oof_logits)
else:
    print('CV disabled. Set CFG["run_cv"] = True for threshold tuning.')

## 10. Optimize class logit biases on OOF

For Macro-F1, small class thresholds can matter a lot. This is learned from train folds only.

In [ ]:
def apply_bias_predict(logits, bias):
    return (logits + np.array(bias).reshape(1, -1)).argmax(axis=1)

best_bias = np.zeros(len(LABELS), dtype=np.float32)
if CFG['run_cv']:
    y_true = train['y'].values
    base_pred = oof_logits.argmax(1)
    base_score = f1_score(y_true, base_pred, average='macro')
    print('OOF no-bias macro F1:', base_score)

    def objective(x):
        # fix technical bias to 0 for identifiability; optimize customer/other relative biases
        bias = np.array([0.0, x[0], x[1]])
        return -f1_score(y_true, apply_bias_predict(oof_logits, bias), average='macro')

    result = differential_evolution(objective, bounds=[(-1.5, 1.5), (-1.5, 1.5)], seed=SEED, tol=1e-4)
    best_bias = np.array([0.0, result.x[0], result.x[1]], dtype=np.float32)
    tuned_pred = apply_bias_predict(oof_logits, best_bias)
    tuned_score = f1_score(y_true, tuned_pred, average='macro')
    print('Best bias:', dict(zip(LABELS, best_bias)))
    print('OOF tuned macro F1:', tuned_score)
    print(classification_report(y_true, tuned_pred, target_names=LABELS))
    print(confusion_matrix(y_true, tuned_pred))

with open(f"{CFG['output_dir']}/best_bias.json", 'w') as f:
    json.dump({lab: float(best_bias[i]) for i, lab in enumerate(LABELS)}, f, indent=2)

## 11. Classical TF-IDF model as a legal small component

This is optional. It can improve short typo-heavy rows. If you are worried about the 600M pipeline interpretation, submit the neural-only file. The classical component is tiny compared with neural models, but document it clearly.

In [ ]:
def build_tfidf_union():
    return FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char', ngram_range=(2,6), min_df=2, max_features=220000, sublinear_tf=True)),
        ('char_wb', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,6), min_df=2, max_features=160000, sublinear_tf=True)),
        ('word', TfidfVectorizer(analyzer='word', token_pattern=r'(?u)\b\w+\b', ngram_range=(1,2), min_df=2, max_features=120000, sublinear_tf=True)),
    ])

run_classical = True
if run_classical:
    tfidf = build_tfidf_union()
    X = tfidf.fit_transform(train['text'])
    Xtest = tfidf.transform(test['text'])
    print(X.shape, Xtest.shape)
    # Calibrated LinearSVC gives probabilities for blending.
    # cv=3 calibration is train-only and acceptable; increase if time allows.
    svc = LinearSVC(C=0.8, class_weight='balanced', random_state=SEED)
    clf = CalibratedClassifierCV(svc, cv=3, method='sigmoid')
    clf.fit(X, train['y'])
    test_tfidf_proba = clf.predict_proba(Xtest)
    np.save(f"{CFG['output_dir']}/test_tfidf_proba.npy", test_tfidf_proba)
    import joblib
    joblib.dump((tfidf, clf), f"{CFG['output_dir']}/tfidf_calibrated_svc.joblib")
else:
    test_tfidf_proba = None

## 12. Train final neural model on all train data

This is the final single checkpoint used for inference.

In [ ]:
final_dir = f"{CFG['output_dir']}/final_full_train"
trainer, tok = train_neural_model(
    train['text'].tolist(), train['y'].tolist(),
    valid_texts=None, valid_y=None,
    out_dir=final_dir,
    seed=SEED,
    epochs=CFG['final_epochs'],
    lr=CFG['final_lr'],
    save_model=bool(CFG.get('save_final_model', False)),
)

# Predict immediately from memory to avoid saving/loading large model shards.
final_test_logits = predict_neural_in_memory(trainer.model, tok, test['text'].tolist(), batch_size=max(CFG['batch_size']*2, 16))
np.save(f"{CFG['output_dir']}/final_test_logits.npy", final_test_logits)

# Free GPU/CPU memory. If save_final_model=False, no model shards are kept on disk.
del trainer, tok
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
if not CFG.get('save_final_model', False) and os.path.exists(final_dir):
    shutil.rmtree(final_dir, ignore_errors=True)


## 13. Optional final weight soup

This trains several full-train checkpoints and averages weights into **one physical checkpoint**. Since discussion around weight averaging may need organizer confirmation, this is OFF by default.

Do not use multiple checkpoints at inference if their total parameters exceed the rule.

In [ ]:
def average_checkpoints(model_dirs, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    # Copy tokenizer/config from first checkpoint.
    for fname in os.listdir(model_dirs[0]):
        src = os.path.join(model_dirs[0], fname)
        dst = os.path.join(out_dir, fname)
        if os.path.isfile(src) and not fname.endswith('.bin') and not fname.endswith('.safetensors'):
            shutil.copy2(src, dst)
    avg_state = None
    for md in model_dirs:
        model = AutoModelForSequenceClassification.from_pretrained(md, torch_dtype=torch.float32, use_safetensors=True)
        sd = model.state_dict()
        if avg_state is None:
            avg_state = {k: v.detach().cpu().clone() for k, v in sd.items()}
        else:
            for k in avg_state:
                avg_state[k] += sd[k].detach().cpu()
        del model
        gc.collect()
    for k in avg_state:
        avg_state[k] /= len(model_dirs)
    model = AutoModelForSequenceClassification.from_pretrained(model_dirs[0], torch_dtype=torch.float32)
    model.load_state_dict(avg_state)
    model.save_pretrained(out_dir, safe_serialization=True)
    tokenizer = AutoTokenizer.from_pretrained(model_dirs[0], use_fast=True)
    tokenizer.save_pretrained(out_dir)
    return out_dir

if CFG['run_weight_soup']:
    if CFG.get('disable_checkpoint_saving', True):
        raise RuntimeError('Weight soup needs saved checkpoints. Set disable_checkpoint_saving=False and ensure enough disk space, or keep run_weight_soup=False.')
    soup_dirs = []
    for seed in CFG['soup_seeds']:
        md = f"{CFG['output_dir']}/full_seed_{seed}"
        trainer, tok = train_neural_model(
            train['text'].tolist(), train['y'].tolist(),
            valid_texts=None, valid_y=None,
            out_dir=md,
            seed=seed,
            epochs=CFG['final_epochs'],
            lr=CFG['final_lr'],
            save_model=True,
        )
        soup_dirs.append(md)
        del trainer, tok
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    soup_dir = f"{CFG['output_dir']}/final_weight_soup"
    average_checkpoints(soup_dirs, soup_dir)
    soup_test_logits = predict_neural(soup_dir, test['text'].tolist(), batch_size=max(CFG['batch_size']*2, 16))
    np.save(f"{CFG['output_dir']}/soup_test_logits.npy", soup_test_logits)
else:
    soup_test_logits = None

## 14. Generate submissions

No manual test-row labeling. All labels here come from learned models and train-only calibration.

In [ ]:
def make_submission(pred_ids, filename):
    sub = sample.copy()
    sub['label'] = [id2label[int(i)] for i in pred_ids]
    assert list(sub.columns) == ['id', 'label']
    assert len(sub) == len(sample)
    assert sub['id'].tolist() == sample['id'].tolist()
    assert set(sub['label']).issubset(set(LABELS))
    assert sub['label'].isna().sum() == 0
    sub.to_csv(filename, index=False)
    print('\nSaved:', filename)
    print(sub['label'].value_counts(normalize=True))
    print(sub['label'].value_counts())
    return sub

# Neural-only, no bias
pred_neural = final_test_logits.argmax(1)
make_submission(pred_neural, 'submission_neural_single_raw.csv')

# Neural-only, OOF-tuned class bias
pred_neural_bias = apply_bias_predict(final_test_logits, best_bias)
make_submission(pred_neural_bias, 'submission_neural_single_oof_bias.csv')

# Optional soup predictions
if soup_test_logits is not None:
    pred_soup = soup_test_logits.argmax(1)
    make_submission(pred_soup, 'submission_neural_weight_soup_raw.csv')
    pred_soup_bias = apply_bias_predict(soup_test_logits, best_bias)
    make_submission(pred_soup_bias, 'submission_neural_weight_soup_oof_bias.csv')

# Optional neural + TFIDF blend. Tune weights with OOF in a serious run; defaults are conservative.
if test_tfidf_proba is not None:
    neural_proba = softmax(final_test_logits + best_bias.reshape(1, -1), axis=1)
    for w in [0.75, 0.85, 0.92]:
        blend = w * neural_proba + (1-w) * test_tfidf_proba
        make_submission(blend.argmax(1), f'submission_neural_tfidf_blend_w{int(w*100)}.csv')

try:
    from google.colab import files
    for fname in ['submission_neural_single_oof_bias.csv', 'submission_neural_tfidf_blend_w85.csv']:
        if os.path.exists(fname): files.download(fname)
except Exception:
    pass

## 15. Final checklist before DataRace upload

- `id,label` columns only.
- Same row count and same ID order as `sample.csv`.
- No index column.
- No API models.
- No test-set training, MLM, pseudo-labeling, or manual test-row labels.
- Final inference pipeline under 600M parameters.
- Document model name, parameter count, training runtime, seed, and whether checkpoint soup/classical blend was used.

Recommended upload order after running this notebook:

1. `submission_neural_single_oof_bias.csv` — safest neural submission.
2. `submission_neural_tfidf_blend_w85.csv` — if classical+neural hybrid is acceptable to you.
3. `submission_neural_weight_soup_oof_bias.csv` — only if you used weight soup and are comfortable defending it as one final checkpoint.